# Exploratory Data Analysis

## Intro

In this section, we discuss our data cleaning and EDA on the data. In particular, we are interested how prices behave, and how much a user's history says about their next game.

In [1]:
from pathlib import Path

import hvplot.pandas
import hvplot.polars
import numpy as np
import plotly.express as px
import plotly.io as pio
import polars as pl
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

from utils import LABELS, bucket, load_games

DATA = Path("data")
pio.renderers.default = "plotly_mimetype+notebook_connected"
PRICE, FREE = "#4c78a8", "#e45756"
BLUES = [f"#{r:02x}{g:02x}{b:02x}" for r, g, b in np.array([np.interp(np.linspace(0, 1, 256), [0, 1], ch)
         for ch in zip((241, 245, 250), (37, 70, 122))]).T.round().astype(int)]


def show(df):
    return df.to_pandas().style.hide(axis="index").format(precision=2, thousands=",", na_rep="")

def with_stats(plot, df, by, order):
    stats = (df.group_by(by)
               .agg(median=pl.col("price").median(), mean=pl.col("price").mean().round(2),
                    q1=pl.col("price").quantile(0.25), q3=pl.col("price").quantile(0.75), reviews=pl.len())
               .with_columns(pl.col(by).cast(pl.Utf8))
               .sort(pl.col(by).replace_strict(list(order), list(range(len(order))), default=len(order))).to_pandas())
    marks = stats.hvplot.scatter(by, "median", hover_cols=["mean", "q1", "q3", "reviews"], color="white",
                                 line_color="black", size=70)
    return (plot * marks).redim.values(**{by: order})


---

## Data

We use the [Steam Video Game and Bundle Data](https://cseweb.ucsd.edu/~jmcauley/datasets.html#steam_data) from Professor McAuley's lab at UCSD [@kang2018sasrec; @wan2018chains; @pathak2017bundles]: reviews, purchases and
bundle information crawled from Steam, collected for research on sequential recommendation, user behavior and
bundle recommendation. We use Version 2, which has 7.8 million reviews (who wrote it, which game, the date,
hours played and the text) up to January 2018, plus metadata for 32,000 games, including each game's list
price. Version 1 (Australian users' libraries and reviews) is left out, because its user ids don't match the numeric ids in Version 2.

A note on the price. Every price here is the list price in the catalogue on the day the data was crawled. It is not the price the reviewer paid, and it leaves out sales and bundle discounts. So when prices look higher in later years below, that can come from which games people review, or from list prices changing, and this data cannot tell the two apart. The same goes for reviews: the data records when someone wrote a review, not when they bought the game or played it, so a "next review" means the next game reviewed, not the next game bought.

Here are the first five raw reviews.

In [2]:
raw = pl.read_parquet(DATA / "cache" / "reviews.parquet")

display(show(raw.head(5)))

Matplotlib is building the font cache; this may take a moment.


username,user_id,product_id,date,hours,products,found_funny,early_access
Chaos Syren,,725280,2017-12-17 00:00:00,0.10,41.00,,False
₮ʜᴇ Wᴀʀᴛᴏɴ,,328100,2017-12-27 00:00:00,51.10,769.00,,False
hello?<,,328100,2017-10-16 00:00:00,14.60,2.00,2.00,False
Cyderine916,,35140,2018-01-04 00:00:00,5.00,64.00,,False
DarklyThinking,76561198007483075,35140,2018-01-04 00:00:00,16.60,577.00,,False


Here are the first five games.

In [3]:
games = load_games(DATA / "steam_games.json")

display(show(games.head(5)))

product_id,title,price,price_raw,genres,release
428605,DARIUSBURST Chronicle Saviours - Fantasy Zone,4.99,4.99,['Action'],2016-04-06 00:00:00
535820,Tom Clancy's Rainbow Six® Siege - Frost The Division,6.99,6.99,['Action'],2016-09-27 00:00:00
294040,Loot Hero DX,2.99,2.99,['Action' 'Adventure' 'Casual' 'Indie' 'RPG'],2015-07-03 00:00:00
551821,Total War: WARHAMMER - Grey Wizard,0.00,Free,['Action' 'Strategy'],2016-12-08 00:00:00
552660,8-Bit Arena VR,0.99,0.99,[],


In [1]:
examples = ["Team Fortress 2", "Undertale", "Stardew Valley", "Fallout 4", "Grand Theft Auto V"]
show(games.filter(pl.col("title").is_in(examples)).select("title", "price").sort("title"))

title,price
Fallout 4,29.99
Grand Theft Auto V,39.99
Stardew Valley,14.99
Team Fortress 2,0.00
Undertale,9.99


---

## Cleaning

We started with 7.8 million reviews and ended up with 2.8 million. The table below shows each step and how many reviews it removed.

In [4]:
dedup = raw.unique()
with_id = dedup.filter(pl.col("user_id").is_not_null())
priced = with_id.join(games.drop_nulls("price").select("product_id", "price", "genres", "title"), on="product_id")
ev = (priced.sort("user_id", "date", "product_id")
            .unique(["user_id", "product_id"], keep="first", maintain_order=True)
            .with_columns(day=pl.col("date").dt.date()))

show(pl.DataFrame({
    "step": ["raw reviews", "remove duplicates", "keep reviews with a user id", "keep games with a price",
             "one review per user and game"],
    "why": ["", "scraped twice", "no user id",
            "no usable price", "same game reviewed twice"],
    "reviews": [raw.height, dedup.height, with_id.height, priced.height, ev.height],
}).with_columns(removed=(pl.col("reviews").shift(1) - pl.col("reviews")).fill_null(0)))

step,why,reviews,removed
raw reviews,,"7,793,069",0
remove duplicates,scraped twice,"6,961,672","831,397"
keep reviews with a user id,no user id,"2,833,192","4,128,480"
keep games with a price,no usable price,"2,806,118","27,074"
one review per user and game,same game reviewed twice,"2,805,451",667


A duplicate is the same review scraped twice: same user, same game, same date, same hours played.

In [5]:
show(raw.filter(pl.col("user_id").is_not_null() & pl.struct(pl.all()).is_duplicated())
        .sort("user_id", "product_id", "date").head(4)
        .select("username", "user_id", "product_id", pl.col("date").dt.date(), "hours"))

username,user_id,product_id,date,hours
The Doctor,76561197960266546,319630,2016-11-25 00:00:00,13.90
The Doctor,76561197960266546,319630,2016-11-25 00:00:00,13.90
Nem3si$,76561197960266564,317400,2016-08-14 00:00:00,8.30
Nem3si$,76561197960266564,317400,2016-08-14 00:00:00,8.30


Most of the cut is reviews without a user id. Those accounts only have a display name, which isn't unique, so we can't link their reviews. We'd rather lose them than mix two people's histories together. The table below checks whether the dropped reviews look different from the kept ones. They are close: the dropped reviews average \$15.34 against \$16.79 for the kept ones, and 18.7% are free against 19.5%. They are a little cheaper, so the kept reviews run slightly pricier than the raw data.

In [6]:
dropped = (dedup.filter(pl.col("user_id").is_null())
             .join(games.drop_nulls("price").select("product_id", "price"), on="product_id"))
print(f"Priced reviews with no user id: {dropped.height:,}, written under {dropped['username'].n_unique():,} display names")
show(pl.DataFrame({
    "reviews": ["dropped (no user id)", "kept (one per user and game)"],
    "mean price": [dropped["price"].mean(), ev["price"].mean()],
    "median price": [dropped["price"].median(), ev["price"].median()],
    "percent free": [100 * (dropped["price"] == 0).mean(), 100 * (ev["price"] == 0).mean()],
}))

Priced reviews with no user id: 4,087,625, written under 1,334,481 display names


reviews,mean price,median price,percent free
dropped (no user id),15.34,13.99,18.73
kept (one per user and game),16.79,14.99,19.54


Free games come in many spellings, and a few prices aren't prices at all.

In [7]:
outcome = (pl.when(pl.col("price_raw").is_null()).then(pl.lit("missing"))
             .when(pl.col("price").is_null()).then(pl.lit("not a price, dropped"))
             .when(pl.col("price") == 0).then(pl.lit("free (any spelling), set to $0"))
             .otherwise(pl.lit("price")))
show(games.group_by(outcome.alias("catalogue price")).agg(games=pl.len(),
    examples=pl.col("price_raw").unique().sort().head(4).str.join(", ")).sort("games", descending=True))

catalogue price,games,examples
price,"28,847","0.49, 0.5, 0.89, 0.95"
"free (any spelling), set to $0","1,900","Free, Free Demo, Free HITMAN™ Holiday Pack, Free Mod"
missing,"1,377",
"not a price, dropped",8,"Install Now, Install Theme, Play Now, Play WARMACHINE: Tactics Demo"


---

## What are we predicting?

We want to predict the price of the next game a user reviews, so we treat each review as a prediction, as long as the user has reviewed something before it. A user's very first review has no history to learn from, so we skip it. Across all users that leaves 1.25 million reviews to predict.

We split by time, not at random. The cutoffs are the 80th and 90th percentiles of all review dates, so about 80% of reviews fall in train, 10% in validation and 10% in test. The cutoffs and the counts are printed in the cell below.

In [8]:
T1, T2 = ev["date"].quantile(0.8), ev["date"].quantile(0.9)
ev = ev.with_columns(split=pl.when(pl.col("date") < T1).then(pl.lit("train"))
                             .when(pl.col("date") < T2).then(pl.lit("validation")).otherwise(pl.lit("test")))
print(f"Cutoffs: train before {T1.date()}, validation before {T2.date()}, test from then on")
print("Reviews per split: " + ", ".join(f"{s} {(ev['split'] == s).sum():,} "
                                       f"({(ev['split'] == s).mean():.0%})" for s in ("train", "validation", "test")))
first_day = pl.col("day").min().over("user_id")
targets = ev.filter(pl.col("day") > first_day)
print(f"Prediction targets (reviews after a user's first): {targets.height:,}")

def summary(df, name):
    return df.select(pl.lit(name).alias("set"), count=pl.len(), mean=pl.col("price").mean(),
                     median=pl.col("price").median(), std=pl.col("price").std(),
                     free=(pl.col("price") == 0).mean(), p90=pl.col("price").quantile(0.9), max=pl.col("price").max())

show(pl.concat([summary(games.drop_nulls("price"), "catalogue (every game once)"), summary(ev, "all reviews"),
                *[summary(targets.filter(pl.col("split") == s), f"{s} targets") for s in ("train", "validation", "test")]]))

Cutoffs: train before 2017-07-07, validation before 2017-11-06, test from then on
Reviews per split: train 2,241,709 (80%), validation 283,129 (10%), test 280,613 (10%)


Prediction targets (reviews after a user's first): 1,252,608


set,count,mean,median,std,free,p90,max
catalogue (every game once),"30,747",8.90,4.99,16.34,0.06,19.99,995.00
all reviews,"2,805,451",16.79,14.99,17.60,0.20,39.99,771.71
train targets,"959,337",16.11,14.99,16.34,0.16,34.99,771.71
validation targets,"150,066",18.86,14.99,17.45,0.17,39.99,771.71
test targets,"143,205",19.46,14.99,16.75,0.14,39.99,771.71


Reviewed games cost more than the catalogue average (\$16.79 vs \$8.90). One possible reason is that popular games get more reviews, but we did not test that. Prices also rise from train (\$16.11) to validation (\$18.86) to test (\$19.46). A model that learns from train's average will guess a little low.

In [9]:
points = (ev.group_by("price").agg(reviews=pl.len()).sort("reviews", descending=True).head(15)
            .with_columns(price_label=pl.format("${}", pl.col("price")), percent=(100 * pl.col("reviews") / ev.height).round(2)))
points.hvplot.bar("price_label", "reviews", color=PRICE, hover_cols=["percent"], rot=45, width=750, height=300,
                  xlabel="list price", ylabel="reviews", yformatter="%.0f", title="Most common prices of reviewed games")

:Bars   [price_label]   (reviews,percent)

Prices aren't spread evenly. Free (19.5% of reviews) and \$19.99 (19.5%) are the two most common prices, and \$9.99 and \$14.99 come next. That spike matters for how we evaluate: guessing \$20 for a free game is a large miss on a log scale, which is why we report MSLE next to MAE and MSE, and why free and paid games get separate attention in the modeling section.

---

## Prices over time

In [10]:
monthly = (ev.sort("date").group_by_dynamic("date", every="1mo")
             .agg(mean_price=pl.col("price").mean(), percent_free=(100 * (pl.col("price") == 0).mean()).round(2), reviews=pl.len())
             .filter(pl.col("date") >= pl.datetime(2013, 1, 1)))
(monthly.hvplot.line("date", "mean_price", color=PRICE, ylabel="mean price ($)", hover_cols=["reviews"], height=250, width=750,
                     title="Mean price of reviewed games per month")
 + monthly.hvplot.line("date", "percent_free", ylabel="% free", hover_cols=["reviews"], height=250, width=750,
                       color=FREE, title="Share of reviews for free games")).cols(1)

:Layout
   .Curve.I  :Curve   [date]   (mean_price,reviews)
   .Curve.II :Curve   [date]   (percent_free,reviews)

Reviewed games do not climb steadily. The average price dips in 2015 (\$14.93) and rises again in 2016 and 2017, reaching \$18.79 in 2017. The free share jumps from 9% in 2013 to 23% in 2015 and eases to 19% by 2017. A random split would mix reviews from different years and hide this. It may also be part of why the test targets are pricier than train's: they average \$19.46 against \$16.11. The test window covers only about two months, and the monthly plot uses all reviews rather than targets, so we have not shown this directly.

In [11]:
by_year_all = (ev.filter(pl.col("day").dt.year().is_between(2013, 2017), pl.col("price") <= 70)
                 .with_columns(year=pl.col("day").dt.year().cast(pl.Utf8)).select("year", "price"))
by_year = by_year_all.sample(150_000, seed=0).to_pandas()
by_year = by_year.sort_values("year", kind="stable")
plot = by_year.hvplot.violin(y="price", by="year", color=PRICE, width=750, height=320, ylabel="price ($)", xlabel="",
                             title="Price of reviewed games by year (games up to $70)")
with_stats(plot, by_year_all, "year", sorted(by_year["year"].unique()))

:Overlay
   .Violin.I  :Violin   [year]   (price)
   .Scatter.I :Scatter   [year]   (median,mean,q1,q3,reviews)

In [12]:
years = (ev.filter(pl.col("day").dt.year().is_between(2013, 2017))
           .group_by(pl.col("day").dt.year().alias("year"))
           .agg(reviews=pl.len(), mean_price=pl.col("price").mean(), median_price=pl.col("price").median(),
                p90=pl.col("price").quantile(0.9), percent_free=100 * (pl.col("price") == 0).mean())
           .sort("year"))
show(years)

year,reviews,mean_price,median_price,p90,percent_free
"2,013","92,392",16.46,14.99,29.99,9.06
"2,014","413,603",16.03,14.99,29.99,15.39
"2,015","556,916",14.93,14.99,29.99,22.66
"2,016","735,930",16.14,14.99,39.99,22.05
"2,017","945,116",18.79,14.99,39.99,19.19


The upper tail gets heavier over time. The 90th percentile is \$29.99 through 2015 and \$39.99 from 2016 on, so the top tenth of reviewed games cost more in later years. (The violins are drawn from a random sample of 150,000 reviews to keep the page light; the white dots and their hover numbers use every review.)

---

## Users

In [13]:
per_user = ev.group_by("user_id").len()
history = targets.join(ev.group_by("user_id", "day").len().sort("user_id", "day")
                          .with_columns(n=pl.col("len").cum_sum().shift(1).over("user_id")).drop("len"),
                       on=["user_id", "day"])
show(pl.DataFrame({
    "": ["users", "median reviews per user", "mean reviews per user", "users with one review",
         "targets with one earlier review", "targets with 5+ earlier reviews"],
    "value": [f"{per_user.height:,}", f"{per_user['len'].median():.0f}", f"{per_user['len'].mean():.1f}",
              f"{(per_user['len'] == 1).mean():.0%}", f"{(history['n'] == 1).mean():.0%}", f"{(history['n'] >= 5).mean():.0%}"],
}))

,value
users,"1,475,626"
median reviews per user,1
mean reviews per user,1.9
users with one review,70%
targets with one earlier review,35%
targets with 5+ earlier reviews,32%


70% of users wrote only one review, and 35% of the reviews we predict have just one earlier review behind them. For those users, their own history can't say much, so we need other signals.

```{anywidget} demos/user_timeline.js
{"url": "user_timelines.json"}
```

Zooming out, every user with at least five reviews is one point below: how much they usually pay, how much that varies, and how often they pick free games. Drag to rotate.

In [14]:
personas = (ev.filter(pl.col("split") == "train").group_by("user_id")
              .agg(reviews=pl.len(), mean_price=pl.col("price").mean().round(2), price_spread=pl.col("price").std().round(2),
                   percent_free=(100 * (pl.col("price") == 0).mean()).round(2))
              .filter(pl.col("reviews") >= 5, pl.col("mean_price") <= 60, pl.col("price_spread") <= 40)
              .sample(8_000, seed=0)
              .with_columns(log_reviews=pl.col("reviews").log10().round(2)))
px.scatter_3d(personas.to_pandas(), x="mean_price", y="price_spread", z="percent_free", color="log_reviews",
              color_continuous_scale="Viridis", opacity=0.8, hover_data=["reviews"], height=620,
              labels={"mean_price": "mean price ($)", "price_spread": "price spread ($)", "percent_free": "% free",
                      "log_reviews": "reviews (log10)"},
              title="Users with 5+ reviews").update_traces(marker_size=2.5)

The cloud tilts two ways. Users who review more free games pay less on average (correlation −0.50), and users with higher average prices also vary more in what they pay (0.66). The plot shows a random 8,000 of the 70,086 training users with five or more reviews, after dropping users whose average price is over \$60 or whose price spread is over \$40 (the count removed by that filter is not printed). Among all 70,086, 43% never reviewed a free game, and those users average \$19.30. Only 0.8% are mostly free, with 90% or more of their reviews free. Most users sit in the middle of the cloud, where the next game could be almost anything. Part of the tilt is mechanical: a free game has price zero, so users with more free games have lower averages by construction, and the spread of a user's prices grows with their mean.

In [15]:
heavy = (ev.filter(pl.col("split") == "train").group_by("user_id")
           .agg(reviews=pl.len(), mean_price=pl.col("price").mean(), spread=pl.col("price").std(),
                free_share=(pl.col("price") == 0).mean())
           .filter(pl.col("reviews") >= 5))
print(f"Users with 5+ train reviews: {heavy.height:,}")
print(f"Correlation of free share with mean price: {heavy.select(pl.corr('free_share', 'mean_price')).item():.2f}")
print(f"Correlation of price spread with mean price: {heavy.select(pl.corr('spread', 'mean_price')).item():.2f}")
never = heavy.filter(pl.col("free_share") == 0)
print(f"Never reviewed a free game: {never.height / heavy.height:.0%}, averaging ${never['mean_price'].mean():.2f}")
print(f"Mostly free (90%+ of reviews): {(heavy['free_share'] >= 0.9).mean():.1%}")

Users with 5+ train reviews: 70,086
Correlation of free share with mean price: -0.50
Correlation of price spread with mean price: 0.66
Never reviewed a free game: 43%, averaging $19.30
Mostly free (90%+ of reviews): 0.8%


---

## Does the last game predict the next one?

The simplest guess for a user's next price is the price of their last game. How well does that work?

In [16]:
prev = (ev.group_by("user_id", "day").agg(day_price=pl.col("price").mean()).sort("user_id", "day")
          .with_columns(prev=pl.col("day_price").shift(1).over("user_id")).select("user_id", "day", "prev"))
trans = (targets.join(prev, on=["user_id", "day"])
                .with_columns(bucket("prev"), bucket("price"))
                .group_by("prev_b", "price_b").len()
                .with_columns(percent=(100 * pl.col("len") / pl.col("len").sum().over("prev_b")).round(2),
                              previous=pl.col("prev_b").replace_strict(range(7), LABELS),
                              next=pl.col("price_b").replace_strict(range(7), LABELS))
                .sort("prev_b", "price_b"))
trans.hvplot.heatmap("next", "previous", "percent", cmap=BLUES, width=560, height=380, hover_cols=["len"], clabel="% of row",
                     flip_yaxis=True, xlabel="next review", ylabel="previous review", title="Price bucket, previous vs next review")

:HeatMap   [next,previous]   (percent,len)

Each row is the price of a user's previous game, and the colors show where their next game landed. Free games often lead to another free game (35% of the time, compared with 16% of prediction targets), but every row is spread out: the last price narrows the range but doesn't set the next one.

In [17]:
free_prev = trans.filter(pl.col("prev_b") == 0)
print(f"Free next after a free game: {free_prev.filter(pl.col('price_b') == 0)['len'].sum() / free_prev['len'].sum():.0%}")
print(f"Free next overall: {trans.filter(pl.col('price_b') == 0)['len'].sum() / trans['len'].sum():.0%}")

Free next after a free game: 35%
Free next overall: 16%


---

## A map of Steam

The data has no geography, but games can still sit near each other. We describe each game only by *who*
reviewed it, so two games end up close together when mostly the same people reviewed both. Price is never used
to place a game; it only sets the colour. The axes have no meaning, only the distances do (each game's list of
reviewers is compressed with SVD, then flattened to 2D with t-SNE). Hover a dot to see the game.

:::{note} How to read the map
Games that share reviewers sit close together. Only nearby dots say much: the gaps between far-apart groups aren't reliable.
:::

In [18]:
train_ev = ev.filter(pl.col("split") == "train")
top = (train_ev.group_by("product_id").agg(reviews=pl.len()).sort("reviews", descending=True).head(2_500)
               .join(games.select("product_id", "title", "price"), on="product_id").with_row_index("g"))
sub = train_ev.join(top.select("product_id", "g"), on="product_id")
sub = (sub.join(sub.group_by("user_id").len().filter(pl.col("len") >= 3).select("user_id").with_row_index("u"), on="user_id"))
counts = sp.csr_matrix((np.ones(sub.height), (sub["g"].to_numpy(), sub["u"].to_numpy())), shape=(top.height, sub["u"].max() + 1))
factors = TruncatedSVD(32, random_state=0).fit_transform(normalize(counts))
xy = TSNE(2, init="pca", perplexity=40, random_state=0).fit_transform(normalize(factors))
RANGES = ["free", "under $5", "$5-10", "$10-20", "$20-40", "$40+"]
COLORS = ["#e45756", "#fde725", "#5ec962", "#21918c", "#3b528b", "#440154"]
steam_map = top.with_columns(x=pl.Series(xy[:, 0]), y=pl.Series(xy[:, 1]), size=pl.col("reviews").log10(),
                             price_range=pl.col("price").cut([0, 4.99, 9.99, 19.99, 39.99], labels=RANGES).cast(pl.Utf8))
(px.scatter(steam_map.to_pandas(), x="x", y="y", color="price_range", size="size", size_max=9, hover_name="title",
            hover_data={"price": True, "reviews": True, "x": False, "y": False, "size": False, "price_range": False},
            category_orders={"price_range": RANGES}, color_discrete_sequence=COLORS, height=650, render_mode="svg",
            labels={"price_range": "price"}, title="The 2,500 most reviewed games, placed by who reviews them")
   .update_xaxes(visible=False).update_yaxes(visible=False)
   .update_layout(plot_bgcolor="white", legend=dict(orientation="h", yanchor="top", y=-0.02, xanchor="left", x=0)))

In [19]:
nn = NearestNeighbors(n_neighbors=11, metric="cosine").fit(factors)
_, idx = nn.kneighbors(factors)
idx = idx[:, 1:]

price = top["price"].to_numpy()
log_price = np.log1p(price)
free = price == 0

corr = np.corrcoef(log_price, log_price[idx].mean(axis=1))[0, 1]
free_nbr_share = free[idx].mean(axis=1)
print(f"Correlation of a game's log price with its 10 neighbours' average: {corr:.2f}")
print(f"Free games: {free_nbr_share[free].mean():.0%} of their neighbours are also free")
print(f"All games: {free.mean():.0%} are free")

Correlation of a game's log price with its 10 neighbours' average: 0.83
Free games: 80% of their neighbours are also free
All games: 15% are free


Even though price played no part in where a game lands, neighbours end up with similar prices. The output above checks
this in the 32-dimensional space the map was built from: how closely a game's price follows its neighbours', and how
often free games sit next to other free games.

So a user's past games tell us which neighbourhood they shop in, and each neighbourhood has its own price level.
That's the idea behind looking at what other users reviewed next, in the next section.

:::{note} What could mislead us
- **List price, not paid price.** Prices are catalogue list prices on the crawl day, with no sales or bundle discounts.
- **Reviews, not purchases.** A next review is the next game reviewed, not the next game bought or played.
- **Dropped reviews.** About 4.1 million priced reviews have no user id and are removed. Those we keep look similar on price, but the sample only covers accounts that have an id.
- **Calendar time.** Prices and several features shift across years. The notebook does not control for time, so the feature comparisons below are descriptive.
:::

---

## Features we explored

If a user's own prices aren't enough, what else do we know before they write a review? We grouped our ideas
into five questions. Several of them can move with calendar time, and prices do too (see Prices over time). Each feature is shown on its own, not controlled for the others or for time, so read them as descriptions rather than causes.

In [20]:
is_f2p = pl.col("genres").list.contains("Free to Play").cast(pl.Int32)
days = (ev.sort("user_id", "day", "product_id")
          .group_by("user_id", "day", maintain_order=True)
          .agg(last_game=pl.col("product_id").last(), reviews=pl.len(), f2p=is_f2p.sum())
          .with_columns(prev_game=pl.col("last_game").shift(1).over("user_id"),
                        gap=(pl.col("day") - pl.col("day").shift(1).over("user_id")).dt.total_days(),
                        f2p_share=(pl.col("f2p").cum_sum().over("user_id") - pl.col("f2p"))
                                  / (pl.col("reviews").cum_sum().over("user_id") - pl.col("reviews")))
          .select("user_id", "day", "prev_game", "gap", "f2p_share"))
ctx = (targets.join(days, on=["user_id", "day"])
              .join(games.select(prev_game="product_id", prev_title="title"), on="prev_game", how="left"))
train_ctx = ctx.filter(pl.col("split") == "train")

def outcome():
    return [pl.len().alias("targets"), pl.col("price").mean().round(2).alias("mean_price"),
            (100 * (pl.col("price") == 0).mean()).round(2).alias("percent_free")]

### What did they just play?

In [21]:
after = (train_ctx.group_by("prev_title").agg(*outcome())
                  .sort("targets", descending=True).head(12).sort("percent_free"))
after.hvplot.barh("prev_title", "percent_free", color=FREE, hover_cols=["mean_price", "targets"], height=380, width=700,
                  xlabel="", ylabel="% of next reviews that are free", title="Next review after the 12 most common last games")

:Bars   [prev_title]   (percent_free,mean_price,targets)

After *Team Fortress 2* or *Robocraft*, more than a third of next reviews are free games; after *Fallout 4* or
*DayZ*, the next game averages over \$20. So the last game carries information that its price alone doesn't.

We can turn that into a number: for the user's last game, the average price of what other users reviewed
next. We measure it on training reviews before 2017 and check it on training reviews from 2017, so we can check whether the pattern carries forward in time.

In [22]:
early = train_ctx.filter(pl.col("day") < pl.date(2017, 1, 1))
later = train_ctx.filter(pl.col("day") >= pl.date(2017, 1, 1))
next_after = (early.group_by("prev_game").agg(n=pl.len(), others_next=pl.col("price").mean())
                   .filter(pl.col("n") >= 20))
check = (later.join(next_after, on="prev_game")
              .with_columns(group=pl.col("others_next").qcut(5, labels=["lowest", "low", "middle", "high", "highest"])
                                   .cast(pl.Enum(["lowest", "low", "middle", "high", "highest"])))
              .group_by("group").agg(*outcome()).sort("group"))
check.hvplot.bar("group", "mean_price", color=PRICE, hover_cols=["percent_free", "targets"], height=280, width=600,
                 xlabel="others' average next price after the user's last game, before 2017 (fifths)",
                 ylabel="mean price ($)", title="Training reviews from 2017")

:Bars   [group]   (mean_price,percent_free,targets)

It holds up on the later reviews: when others moved on to pricier games, so does the user. This is the Markov chain
idea from class, and it doesn't need the user to have much history of their own.

### What do they usually play?

In [23]:
bins = ["none", "up to 25%", "25-50%", "50-99%", "all"]
taste = (
    train_ctx.with_columns(
        group=pl.col("f2p_share").cut([0, 0.25, 0.5, 0.99], labels=bins).cast(pl.Enum(bins)))
    .group_by("group").agg(*outcome()).sort("group")
)
taste.hvplot.bar(
    "group", "percent_free", hover_cols=["mean_price", "targets"], height=280, width=600,
    color=FREE, xlabel="share of past games tagged Free to Play",
    ylabel="% of next reviews that are free", title="Free-to-play history")

:Bars   [group]   (percent_free,mean_price,targets)

Users whose history is all free-to-play review a free game next 42.5% of the time, against 12% for users with none. The share is not strictly increasing across the buckets: users whose history is 50–99% free review a free game next 45.8% of the time, slightly more than the all-free group. We did not test whether that gap is noise. Genres of past games may describe taste better than an average price does, but we did not compare the two directly.

Genre is only part of it. How popular and how new a user's past games were also says something about their next one.

In [24]:
F2P = ["none", "up to 25%", "25-50%", "50-99%", "all"]
spread_all = (train_ctx.filter(pl.col("price") <= 70)
                       .with_columns(group=pl.col("f2p_share").cut([0, 0.25, 0.5, 0.99], labels=F2P)).select("group", "price"))
spread = spread_all.sample(150_000, seed=0).to_pandas()
spread["group"] = spread["group"].astype("category").cat.reorder_categories(F2P, ordered=True)
spread = spread.sort_values("group", kind="stable")
plot = spread.hvplot.violin(y="price", by="group", color=FREE, width=750, height=320, xlabel="share of past games tagged Free to Play",
                            ylabel="price of the next game ($)", title="Next price by free-to-play history (games up to $70)")
with_stats(plot, spread_all, "group", F2P)

:Overlay
   .Violin.I  :Violin   [group]   (price)
   .Scatter.I :Scatter   [group]   (median,mean,q1,q3,reviews)

The violins show where that free share comes from: as the history gets more free-to-play, a fat bulge grows at \$0, while the paid part looks similar by eye. We have not printed a paid-only statistic for that.

In [25]:
past = (ev.join(games.select("product_id", "release"), on="product_id", how="left")
          .with_columns(pop=pl.int_range(pl.len()).over("product_id", order_by="date").log1p(),
                        age=(pl.col("day") - pl.col("release")).dt.total_days() / 365)
          .sort("user_id", "day", "product_id")
          .group_by("user_id", "day", maintain_order=True)
          .agg(k=pl.len(), pop=pl.col("pop").sum(), age=pl.col("age").fill_null(0).sum(), k_age=pl.col("age").is_not_null().sum())
          .with_columns(pop_mean=(pl.col("pop").cum_sum().over("user_id") - pl.col("pop"))
                                 / (pl.col("k").cum_sum().over("user_id") - pl.col("k")),
                        age_mean=(pl.col("age").cum_sum().over("user_id") - pl.col("age"))
                                 / (pl.col("k_age").cum_sum().over("user_id") - pl.col("k_age")))
          .select("user_id", "day", "pop_mean", "age_mean"))
past_ctx = train_ctx.join(past, on=["user_id", "day"])
FIFTHS = ["lowest", "low", "middle", "high", "highest"]
AGES = ["under 3 months", "3-12 months", "1-3 years", "3+ years"]
popular = (past_ctx.with_columns(group=pl.col("pop_mean").qcut(5, labels=FIFTHS).cast(pl.Enum(FIFTHS)))
                   .group_by("group").agg(*outcome()).sort("group"))
newness = (past_ctx.filter(pl.col("age_mean").is_finite())
                   .with_columns(group=pl.col("age_mean").cut([0.25, 1, 3], labels=AGES).cast(pl.Enum(AGES)))
                   .group_by("group").agg(*outcome()).sort("group"))
(popular.hvplot.bar("group", "percent_free", color=FREE, hover_cols=["mean_price", "targets"], height=280, width=600,
                    xlabel="popularity of past games (fifths)", ylabel="% of next reviews that are free", title="Popular past games")
 + newness.hvplot.bar("group", "mean_price", color=PRICE, hover_cols=["percent_free", "targets"], height=280, width=600,
                      xlabel="typical age of past games when reviewed", ylabel="mean price ($)", title="New releases")).opts(shared_axes=False).cols(1)

:Layout
   .Bars.I  :Bars   [group]   (percent_free,mean_price,targets)
   .Bars.II :Bars   [group]   (mean_price,percent_free,targets)

Users who review the biggest hits are twice as likely to review a free game next (23% against 11%). One possible reason is that many of the most reviewed games are free-to-play; we did not test that. Users who review games soon after release pay a little more next time (\$17.13 against \$15.31 for users who review old games).

In [26]:
GAPS = ["next day", "2-7 days", "8-30 days", "1-6 months", "6+ months"]
gaps_all = (train_ctx.filter(pl.col("price") <= 70)
                     .with_columns(group=pl.col("gap").cut([1, 7, 30, 180], labels=GAPS)).select("group", "price"))
gaps = gaps_all.sample(150_000, seed=0).to_pandas()
gaps["group"] = gaps["group"].astype("category").cat.reorder_categories(GAPS, ordered=True)
gaps = gaps.sort_values("group", kind="stable")
plot = gaps.hvplot.box(y="price", by="group", color=PRICE, width=750, height=320, xlabel="time since the user's previous review",
                       ylabel="price of the next game ($)", title="Next price by time between reviews (games up to $70)").opts(outlier_alpha=0)
with_stats(plot, gaps_all, "group", GAPS)

:Overlay
   .BoxWhisker.I :BoxWhisker   [group]   (price)
   .Scatter.I    :Scatter   [group]   (median,mean,q1,q3,reviews)

In [27]:
m, d = pl.col("day").dt.month(), pl.col("day").dt.day()
in_sale = (((m == 12) & (d >= 21)) | ((m == 1) & (d <= 4)) | ((m == 6) & (d >= 20)) | ((m == 7) & (d <= 5))
           | ((m == 11) & d.is_between(22, 30)))
show(train_ctx.group_by(pl.when(in_sale).then(pl.lit("during a Steam sale")).otherwise(pl.lit("rest of the year")).alias("when"))
              .agg(*outcome()).sort("when"))

when,targets,mean_price,percent_free
during a Steam sale,"186,746",17.10,11.93
rest of the year,"772,591",15.88,17.51


Users who come back after a long break review pricier games. Their next game averages \$12.78 when they review the next day, and \$17.80 after six months or more. During the big Steam sales (summer, autumn and winter, roughly dated), the share of free games drops from 17.5% to 11.9%. That fits paid games being reviewed more often when they're discounted, but the sale windows are rough, and we haven't tested that directly.

In [28]:
trending = (ev.group_by("day", "product_id").agg(n=pl.len(), price=pl.col("price").first())
              .sort(["day", "n"], descending=[False, True]).group_by("day", maintain_order=True).head(10)
              .group_by("day").agg(trend_price=(pl.col("price") * pl.col("n")).sum() / pl.col("n").sum())
              .with_columns(day=pl.col("day") + pl.duration(days=1)))
daily = (train_ctx.join(trending, on="day").group_by("day")
                  .agg(trend_price=pl.col("trend_price").first(), mean_price=pl.col("price").mean(), targets=pl.len())
                  .filter(pl.col("targets") >= 100)
                  .with_columns(year=pl.col("day").dt.year().cast(pl.Utf8), trend_price=pl.col("trend_price").round(2),
                                mean_price=pl.col("mean_price").round(2)))
daily.hvplot.scatter("trend_price", "mean_price", by="year", hover_cols=["day", "targets"], alpha=0.6, size=12,
                     width=700, height=420, xlabel="price of yesterday's 10 most reviewed games ($)",
                     ylabel="mean price of today's reviews ($)", title="One dot per day (days with 100+ reviews)")

:NdOverlay   [year]
   :Scatter   [trend_price]   (mean_price,day,targets)

In [29]:
print(f"Days plotted: {daily.height:,}")
print("Correlation of yesterday's top-10 price with today's mean price: "
      f"{daily.select(pl.corr('trend_price', 'mean_price')).item():.2f}")
show(daily.group_by("year").agg(days=pl.len(), trend_price=pl.col("trend_price").mean(),
                                mean_price=pl.col("mean_price").mean()).sort("year"))

Days plotted: 1,333
Correlation of yesterday's top-10 price with today's mean price: 0.68


year,days,trend_price,mean_price
2011,3,15.66,12.93
2012,5,15.75,14.07
2013,44,19.18,15.76
2014,364,17.28,15.51
2015,364,14.23,14.67
2016,366,16.98,15.87
2017,187,23.13,17.91


Each dot is one day. Across the 1,333 days we plot, the correlation between yesterday's top-10 price and today's mean price is 0.68. The averages across the 2017 days are highest: yesterday's top games cost \$23.13 and today's reviews \$17.91. These are per-day averages, not the yearly means in the table above. The correlation pools 2011 to 2017, so part of it may come from the shared upward trend. This is not about the user, but it is still associated with the price they see next.

### Who are they?

Steam IDs are handed out in order, so a larger number usually means a newer account. We don't have account creation dates, so the deciles below are a stand-in for account age.

In [30]:
acct = (targets.filter(pl.col("split") == "train")
               .with_columns(account=(pl.col("user_id").cast(pl.Int64) - 76561197960265728))
               .with_columns(decile=pl.col("account").qcut(10, labels=[f"{i}" for i in range(1, 11)]))
               .group_by("decile").agg(mean_price=pl.col("price").mean().round(2), percent_free=(100 * (pl.col("price") == 0).mean()).round(2),
                                       targets=pl.len())
               .sort(pl.col("decile").cast(pl.Int32)))
(acct.hvplot.bar("decile", "mean_price", color=PRICE, hover_cols=["targets"], height=250, width=325, ylabel="mean price ($)",
                 xlabel="account age decile (1 = oldest)", title="Mean price")
 + acct.hvplot.bar("decile", "percent_free", hover_cols=["targets"], height=250, width=325, color=FREE,
                   ylabel="% free", xlabel="account age decile (1 = oldest)", title="Free share"))

:Layout
   .Bars.I  :Bars   [decile]   (mean_price,targets)
   .Bars.II :Bars   [decile]   (percent_free,targets)

Newer accounts review more free games (27% vs 9% for the oldest), a pattern we didn't expect to find in an id.

How long they've been reviewing matters too, separately from how old the account is.

In [31]:
TENURE = ["first month", "1-6 months", "6-12 months", "1-2 years", "2+ years"]
first = ev.group_by("user_id").agg(first_day=pl.col("day").min())
tenure = (train_ctx.join(first, on="user_id")
                   .with_columns(group=(pl.col("day") - pl.col("first_day")).dt.total_days()
                                       .cut([30, 180, 365, 730], labels=TENURE).cast(pl.Enum(TENURE)))
                   .group_by("group").agg(*outcome()).sort("group"))
tenure.hvplot.bar("group", "mean_price", color=PRICE, hover_cols=["percent_free", "targets"], height=280, width=600,
                  xlabel="time since the user's first review", ylabel="mean price ($)", title="Time since first review")

:Bars   [group]   (mean_price,percent_free,targets)

In [32]:
gap_tbl = (train_ctx.with_columns(group=pl.col("gap").cut([1, 7, 30, 180], labels=GAPS).cast(pl.Enum(GAPS)))
           .group_by("group").agg(*outcome()).sort("group"))
for name, t in {"Free-to-play history (share of past games tagged Free to Play)": taste,
                "Popularity of past games (fifths)": popular,
                "Age of past games": newness,
                "Account age (decile 1 is oldest)": acct,
                "Time since first review": tenure,
                "Next review after the 12 most common last games": after,
                "Others' next price after the user's last game (fifths)": check,
                "Time between reviews": gap_tbl, "Price points of reviewed games": points}.items():
    print(name)
    display(show(t))

Free-to-play history (share of past games tagged Free to Play)


group,targets,mean_price,percent_free
none,"637,584",17.42,11.62
up to 25%,"129,455",15.05,11.85
25-50%,"87,969",14.13,26.19
50-99%,"23,292",10.02,45.75
all,"81,037",11.42,42.49


Popularity of past games (fifths)


group,targets,mean_price,percent_free
lowest,"191,868",14.58,10.87
low,"191,867",15.81,13.81
middle,"191,867",16.55,15.87
high,"191,867",16.99,18.37
highest,"191,868",16.63,23.21


Age of past games


group,targets,mean_price,percent_free
under 3 months,"170,737",17.13,18.61
3-12 months,"188,383",16.77,15.73
1-3 years,"361,118",15.82,15.65
3+ years,"237,431",15.31,16.53


... (8 outputs left out)

Time between reviews


group,targets,mean_price,percent_free
next day,"40,968",12.78,13.53
2-7 days,"103,471",13.94,14.86
8-30 days,"173,817",15.01,16.33
1-6 months,"350,448",16.29,17.11
6+ months,"290,633",17.80,16.63


Price points of reviewed games


price,reviews,price_label,percent
0.00,"548,161",$0.0,19.54
19.99,"546,695",$19.99,19.49
9.99,"354,169",$9.99,12.62
14.99,"303,080",$14.99,10.80
29.99,"204,751",$29.99,7.30
39.99,"165,660",$39.99,5.90
4.99,"117,647",$4.99,4.19
59.99,"93,091",$59.99,3.32
24.99,"91,962",$24.99,3.28
0.99,"36,427",$0.99,1.30


Users in their first month of reviewing average \$14.17 per game, and 20% of their next reviews are free games. After two years of reviewing, it's \$17.69 and 12%.